<a href="https://colab.research.google.com/github/Divyanshu8410/NLP/blob/main/23_Implementing%20Attention%20Mechanism%20(Basic%20custom%20model).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
pip install tensorflow numpy

In [4]:

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Dense, GlobalAveragePooling1D, Layer

# 1. Generate sample sequence dataset
np.random.seed(42)

X = np.random.randint(1, 20, (1000, 10))
y = (np.sum(X, axis=1) > 100).astype(int)

X_train, X_test = X[:800], X[800:]
y_train, y_test = y[:800], y[800:]

# 2. Custom Attention Layer
class AttentionLayer(Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(
            name="attention_weight",
            shape=(input_shape[-1], 1),
            initializer="glorot_uniform",
            trainable=True
        )
        self.b = self.add_weight(
            name="attention_bias",
            shape=(1,),
            initializer="zeros",
            trainable=True
        )
        super().build(input_shape)

    def call(self, inputs):
        # Calculate attention scores
        scores = tf.nn.tanh(tf.matmul(inputs, self.W) + self.b)

        # Calculate attention weights
        weights = tf.nn.softmax(scores, axis=1)

        # Calculate context vector
        context = tf.reduce_sum(inputs * weights, axis=1)

        return context

# 3. Build Custom Attention Model
input_layer = Input(shape=(10,))

embedding = Embedding(
    input_dim=20,
    output_dim=16
)(input_layer)

attention = AttentionLayer()(embedding)

dense = Dense(16, activation='relu')(attention)
output = Dense(1, activation='sigmoid')(dense)

model = Model(inputs=input_layer, outputs=output)

# 4. Compile Model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# 5. Train Model
print("Training Attention Model...")

model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

# 6. Evaluate Model
loss, accuracy = model.evaluate(X_test, y_test)

print("\nTest Accuracy:", round(accuracy * 100, 2), "%")

# 7. Predict New Sequence
new_sequence = np.array([[1, 5, 8, 12, 15, 3, 7, 10, 18, 2]])

prediction = model.predict(new_sequence, verbose=0)

print("\nInput Sequence:", new_sequence[0])
print("Predicted Class:", 1 if prediction[0][0] >= 0.5 else 0)

# 8. Display Model Summary
print("\nModel Architecture:")
model.summary()


Training Attention Model...
Epoch 1/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 32ms/step - accuracy: 0.6984 - loss: 0.6905 - val_accuracy: 0.8687 - val_loss: 0.6854
Epoch 2/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8828 - loss: 0.6780 - val_accuracy: 0.9375 - val_loss: 0.6699
Epoch 3/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9406 - loss: 0.6566 - val_accuracy: 0.9312 - val_loss: 0.6455
Epoch 4/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.9297 - loss: 0.6242 - val_accuracy: 0.9312 - val_loss: 0.6096
Epoch 5/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.9344 - loss: 0.5797 - val_accuracy: 0.9250 - val_loss: 0.5608
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8800 - loss: 0.5688 

Test Accuracy: 88.0 %

Input Sequence: [ 1  5  8 12 15  3  7 10 18  2]
Predicted Class: 0

Model Architecture:


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 10)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_2 (Embedding)         │ (None, 10, 16)         │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attention_layer                 │ (None, 16)             │            17 │
│ (AttentionLayer)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,880 (7.35 KB)

 Trainable params: 626 (2.45 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,254 (4.90 KB)